# Rainfall Pattern & Crop Suitability Analyser

This notebook summarizes illustrative January–December rainfall, compares regional rainfall patterns, screens monthly crop suitability, detects seasonal peaks, and provides a farmer advisory. Reusable classes and calculations are in [`src/rainfall.py`](src/rainfall.py).

## Important interpretation note

The rainfall values below are illustrative monthly values, not multi-year climate normals. Crop bands in this exercise are simple screening rules, not planting recommendations or locally validated agronomic thresholds. Rainfall received is not identical to water available to a crop: soil, runoff, drainage, evapotranspiration, crop stage and rainfall reliability matter. FAO Irrigation Water Management training materials explain that crop water needs are determined monthly, crop type changes water demand, and only part of rainfall is effective ([FAO, *Irrigation Water Management: Irrigation Water Needs*, Chapters 2–4](https://www.fao.org/4/s2022e/s2022e02.htm); [effective rainfall](https://www.fao.org/4/s2022e/s2022e03.htm)). We use those sources to motivate transparent, crop-differentiated screening bands; the specific monthly cutoffs are illustrative and must be locally calibrated.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine

from src.rainfall import CropRule, MONTHS, RainfallAnalyzer, Region

RAINFALL_DATA = {
    "Kampala": [120, 140, 180, 200, 220, 180, 90, 70, 60, 100, 110, 130],
    "Gulu": [8, 25, 75, 160, 190, 145, 170, 215, 175, 150, 60, 15],
    "Mbarara": [70, 85, 120, 140, 90, 25, 20, 55, 100, 125, 120, 90],
}
regions = [Region(name, values) for name, values in RAINFALL_DATA.items()]
region_by_name = {region.name: region for region in regions}
print(f"{'Month':12} " + " ".join(f"{region.name:>10}" for region in regions))
for index, month in enumerate(MONTHS):
    values = " ".join(f"{region.rainfall[index]:10.0f}" for region in regions)
    print(f"{month:12} {values}")

## 1. Regional annual and monthly summaries

Annual total is the sum of the twelve monthly values. Coefficient of variation (CV) is population standard deviation divided by mean monthly rainfall; a larger CV indicates greater variation relative to the region's own average.

In [ ]:
summary_rows = []
for region in regions:
    summary_rows.append(
        {
            "Region": region.name,
            "Annual total (mm)": region.annual_total(),
            "Monthly mean (mm)": region.mean(),
            "Wettest month": region.wettest_month(),
            "Driest month": region.driest_month(),
            "Monthly rainfall CV": region.coefficient_of_variation(),
        }
    )
print(f"{'Region':10} {'Annual total (mm)':>18} {'Mean (mm)':>12} {'Wettest':>12} {'Driest':>12} {'CV':>8}")
for row in summary_rows:
    print(
        f"{row['Region']:10} {row['Annual total (mm)']:18.1f} "
        f"{row['Monthly mean (mm)']:12.1f} {row['Wettest month']:>12} "
        f"{row['Driest month']:>12} {row['Monthly rainfall CV']:8.3f}"
    )

## 2. Crop screening rules

For this demonstration, maize is assigned a 100–250 mm/month screening band, beans 50–150 mm/month, and coffee 100–200 mm/month. The broader maize band reflects its higher water demand relative to some lower-water crops noted in FAO guidance; beans use a lower band and coffee a moderate-to-high band for contrast. These exact cutoffs are analyst-selected teaching thresholds, not values claimed to be prescribed by FAO. Boundary values count as "Good"; below the minimum is "Drought risk," and above the maximum is flagged as "Waterlogging risk." High monthly rainfall alone does not prove waterlogging—soil drainage and effective rainfall must be assessed.

In [ ]:
crop_rules = (
    CropRule("maize", 100, 250),
    CropRule("beans", 50, 150),
    CropRule("coffee", 100, 200),
)
print("Illustrative crop screening bands (mm/month):")
for crop in crop_rules:
    print(f"  {crop.name.title():8}: {crop.minimum_mm:.0f}–{crop.maximum_mm:.0f}")

### Classify every month in every region

The detailed table preserves the categorical result for every region × crop × month. This is a rainfall-only screen and should be interpreted together with planting dates, crop growth stage and field drainage.

In [ ]:
classification_rows = []
for region in regions:
    for crop in crop_rules:
        for month, rainfall_mm in zip(MONTHS, region.rainfall):
            classification_rows.append(
                {
                    "Region": region.name,
                    "Crop": crop.name.title(),
                    "Month": month,
                    "Rainfall (mm)": rainfall_mm,
                    "Classification": crop.classify(float(rainfall_mm)),
                }
            )
for region in regions:
    print(f"\n{region.name}")
    for crop in crop_rules:
        labels = [crop.classify(float(value)) for value in region.rainfall]
        print(f"  {crop.name.title():8}: " + "; ".join(
            f"{month}: {label}" for month, label in zip(MONTHS, labels)
        ))

## 3. Correct cosine similarity

`math.cos()` accepts an angle in radians and returns its cosine; it does not compare rainfall vectors. Vector cosine similarity is `dot(a, b) / (||a|| × ||b||)`. SciPy's `scipy.spatial.distance.cosine` returns cosine *distance*, so similarity should equal `1 - distance`. We verify that identity numerically below.

In [ ]:
kampala = region_by_name["Kampala"]
gulu = region_by_name["Gulu"]
numpy_similarity = RainfallAnalyzer.cosine_similarity(kampala, gulu)
scipy_similarity = 1 - cosine(kampala.rainfall, gulu.rainfall)
print(f"NumPy vector cosine similarity: {numpy_similarity:.6f}")
print(f"1 − SciPy cosine distance:      {scipy_similarity:.6f}")
np.testing.assert_allclose(numpy_similarity, scipy_similarity)

Cosine similarity measures direction/shape and ignores vector magnitude: if one region receives exactly twice as much rain in every month, the vectors still have cosine similarity 1. Therefore also inspect Pearson correlation (linear co-movement after centering) and Euclidean distance (absolute differences in mm); cosine similarity alone cannot indicate equal rainfall totals.

## 4. Pairwise regional comparison matrices

Each matrix compares all regions with all others. Cosine similarity and Pearson correlation are dimensionless; Euclidean distance is in millimetres across the 12-month vectors. The diagonal is 1 for similarities/correlations and 0 for distance.

In [ ]:
similarity_matrices = RainfallAnalyzer.similarity_matrices(regions)
for title, (names, matrix) in similarity_matrices.items():
    print(f"\n{title}")
    print(" " * 12 + " ".join(f"{name:>12}" for name in names))
    for name, row in zip(names, matrix):
        print(f"{name:12} " + " ".join(f"{value:12.3f}" for value in row))

## 5. Detect rainy-season peaks

`scipy.signal.find_peaks` detects local maxima. Monthly rainfall is cyclic, so the implementation repeats the annual series three times and keeps peaks from the middle copy; this allows December/January boundary peaks to be detected. A 15 mm prominence and three-month minimum separation suppress minor fluctuations. With only one illustrative annual cycle, the number of peaks is sensitive to these settings and does not establish a regional climate classification.

In [ ]:
season_rows = []
for region in regions:
    peak_months, mode = RainfallAnalyzer.rainy_season_peaks(
        region, prominence_mm=15, minimum_separation_months=3
    )
    season_rows.append(
        {"Region": region.name, "Peak month(s)": ", ".join(peak_months), "Detected mode": mode}
    )
print(f"{'Region':10} {'Peak month(s)':25} {'Detected mode':15}")
for row in season_rows:
    print(f"{row['Region']:10} {row['Peak month(s)']:25} {row['Detected mode']:15}")

### Compare the peaks with Uganda's broad rainfall zones

Uganda is commonly described as having two rainy seasons in much of the south/central region and a more unimodal rainfall regime toward the north. The detector labels the supplied illustrative series by local maxima, not by rainy-season onset/cessation. With the stated prominence settings, Kampala is detected as unimodal, Gulu bimodal and Mbarara bimodal: only Mbarara aligns with that broad expectation. This mismatch is a useful warning: the prompt's single illustrative monthly series is not a long-term climatology, and one-year noise, geography and peak settings can disagree with a climate-zone label. Validate classifications using multi-year monthly normals and authoritative local meteorological data.

## 6. Plot monthly rainfall by region

This line chart compares monthly rainfall patterns. All three regions share the same millimetre scale, so magnitude differences remain visible.

In [ ]:
figure, axis = plt.subplots(figsize=(11, 6))
for region in regions:
    axis.plot(MONTHS, region.rainfall, marker="o", linewidth=2, label=region.name)
axis.set_xlabel("Month")
axis.set_ylabel("Rainfall (mm/month)")
axis.set_title("Illustrative monthly rainfall by region")
axis.grid(alpha=0.25)
axis.legend()
axis.tick_params(axis="x", rotation=35)
figure.tight_layout()
plt.show()

## 7. Crop suitability heatmaps

Each panel maps the monthly crop classification to an ordinal screening score: drought risk = −1, good = 0, and waterlogging risk = +1. The colors encode categories for quick comparison, not a continuous crop-yield estimate. Numeric rainfall labels are shown in every region-month cell.

In [ ]:
score_for = {"Drought risk": -1, "Good": 0, "Waterlogging risk": 1}
figure, axes = plt.subplots(len(crop_rules), 1, figsize=(13, 8), sharex=True)
for axis, crop in zip(axes, crop_rules):
    score_matrix = np.zeros((len(regions), len(MONTHS)), dtype=int)
    annotations = np.empty_like(score_matrix, dtype=object)
    for region_index, region in enumerate(regions):
        for month_index, rainfall_mm in enumerate(region.rainfall):
            label = crop.classify(float(rainfall_mm))
            category = "Good" if label.startswith("Good") else label
            score_matrix[region_index, month_index] = score_for[category]
            annotations[region_index, month_index] = f"{rainfall_mm:.0f} mm"
    image = axis.imshow(score_matrix, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
    for row_index in range(len(regions)):
        for column_index in range(len(MONTHS)):
            axis.text(column_index, row_index, annotations[row_index, column_index],
                      ha="center", va="center", fontsize=8)
    axis.set_yticks(range(len(regions)), labels=[region.name for region in regions])
    axis.set_xticks(range(len(MONTHS)), labels=MONTHS)
    axis.set_title(f"{crop.name.title()} monthly rainfall screen ({crop.minimum_mm:.0f}–{crop.maximum_mm:.0f} mm/month)")
    axis.set_xlabel("")
    axis.set_ylabel("Region")
    axis.tick_params(axis="x", rotation=35)
axes[-1].set_xlabel("Month")
figure.tight_layout()
plt.show()

## 8. Farmer advisory: Gulu

**Illustrative advisory (under 200 words).** For Gulu, the supplied rainfall series shows the strongest rains in May and August, with relatively dry conditions in January, February, November and December. Under the screening bands used here, beans have suitable rainfall more often across the middle of the year than maize or coffee, while the driest months carry drought risk for all three crops. Farmers considering beans could prepare land ahead of the rains and time planting near the onset of a reliable wet period rather than relying on a single calendar month. Avoid waterlogged fields during high-rainfall months; use drainage where needed and select well-drained soils. This is not a district planting calendar: consult current seasonal forecasts, local extension staff, soil moisture and seed-variety guidance before planting. The input series is illustrative and the monthly thresholds are teaching assumptions, not a guarantee of crop performance.

## Conclusions and data limitations

- Region summaries, crop screens, pairwise measures and rainy-season peaks are computed from the supplied illustrative twelve monthly values.
- Correct vector cosine similarity is a normalized dot product and agrees with one minus SciPy's cosine distance. It emphasizes seasonal shape while discarding magnitude.
- The peak detector identifies local maxima; Kampala and Gulu outputs do not both match the broad expected Uganda zones, highlighting that illustrative single-year data should not be treated as climate normals.
- Rainfall screening bands and farmer advice are illustrative. Rainfall is not the same as effective crop water, and soil, crop stage, rainfall reliability and local agronomy matter.

### References

1. FAO, *Irrigation Water Management: Irrigation Water Needs*, Chapter 2 (climate and crop effects on crop water needs): https://www.fao.org/4/s2022e/s2022e02.htm
2. FAO, *Irrigation Water Management: Irrigation Water Needs*, Chapter 3 (effective rainfall, runoff and deep percolation): https://www.fao.org/4/s2022e/s2022e03.htm

The requested multi-year NASA POWER/UNMA extension is not included; these inputs are the specific illustrative series provided in the project brief.